In [9]:
%pip install pandas numpy scikit-learn matplotlib seaborn wordcloud gensim

  Using cached gensim-4.4.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
  Using cached wrapt-2.1.2-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (7.4 kB)
Using cached gensim-4.4.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (27.9 MB)
Using cached wrapt-2.1.2-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (121 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gensim]2m2/3 [gensim]
Note: you may need to restart the kernel to use updated packages.


# Εξόρυξη Γνώσης από Μουσικά Δεδομένα (Audio & Lyrics)

**Ομάδα:**
* Παναγιώτης Μακαρόνας (AM: sdi2300107)
* Αχιλλέας Νιανιακούδης-Κοέν (AM: sdi2300138)

In [10]:

# Main data handling libraries
import pandas as pd     # Used for dataframes handling and data manipulation
import numpy as np      # Main math library in Python

import tarfile      # Opening tarfile to read its insides
import gensim       # NLP (Natural Language Processing)

# The MLTs (Machine Learning Tools)
from sklearn.cluster import KMeans      # Great for finding clusters and grouping them
from sklearn.decomposition import PCA   # Lowering data size to help the process
from sklearn.manifold import TSNE       # Helping with organizing massive data groups for plots
from sklearn.metrics.pairwise import cosine_similarity  # Finding the similarity between two songs

# Drawing Tools
import matplotlib.pyplot as plt     # Generating plots
import seaborn as sns               # Helps in making the plots more modern
from wordcloud import WordCloud     # Image generator for most frequently used tags in a cluster


## 1. Συλλογή Δεδομένων (Data collection)
Σε αυτό το κελί φορτώνονται τα δεδομένα, εφαρμόζεται το φιλτράρισμα στα Top-5 Genres και πραγματοποιείται το intersection των αρχείων.

In [1]:

# File paths
genres_path = "data/id_genres.csv"
info_path = "data/id_information.csv"
mfcc_path = "data/id_mfcc_stats.tsv.bz2"
tags_path = "data/id_tags.csv"
lyrics_archive = "data/processed_lyrics.tar.gz"

# Load CSVs
df_genres = pd.read_csv(genres_path)
df_info = pd.read_csv(info_path)
df_tags = pd.read_csv(tags_path)
df_mfcc = pd.read_csv(mfcc_path, sep='\t', compression='bz2')

# Top-5 genres filtering
top5 = df_genres['genre'].value_counts().head(5).index.tolist()
df_genres = df_genres[df_genres['genre'].isin(top5)]
print(f"Top-5 genres: {top5}")

# Intersection: keep only IDs present in genres, mfcc AND lyrics archive
genre_ids = set(df_genres['id'])
mfcc_ids = set(df_mfcc.iloc[:, 0])  # first col is the song id
common_ids = genre_ids & mfcc_ids   # songs in both genres and mfcc

# Extract lyrics only for common IDs
lyrics_dict = {}
with tarfile.open(lyrics_archive, 'r:gz') as tar:
    for member in tar:
        if not member.isfile():
            continue
        song_id = member.name.split('/')[-1].replace('.txt', '')
        if song_id.isdigit() and int(song_id) in common_ids:
            f = tar.extractfile(member)
            if f:
                lyrics_dict[int(song_id)] = f.read().decode('utf-8', errors='ignore')

# Build final clean DataFrame
df_lyrics = pd.DataFrame(list(lyrics_dict.items()), columns=['id', 'lyrics'])
final_ids = common_ids & set(df_lyrics['id'])

df = df_genres[df_genres['id'].isin(final_ids)].merge(df_lyrics, on='id')
df = df.merge(df_mfcc, left_on='id', right_on=df_mfcc.columns[0])

print(f"Final dataset: {len(df)} songs across {df['genre'].nunique()} genres")
print(df[['id', 'genre', 'lyrics']].head())


NameError: name 'pd' is not defined

## 2. Εξαγωγή Χαρακτηριστικών & Embeddings (Feature Extraction)
Εδώ δημιουργούνται οι διανυσματικές αναπαραστάσεις για το κείμενο (Word2Vec/Doc2Vec) και τον ήχο (PCA στα MFCCs).

In [ ]:

# =============================================
# 2a. TEXT EMBEDDINGS  (Word2Vec → avg vector)
# =============================================

# Tokenize lyrics into word lists
from gensim.models import Word2Vec

tokenized = df['lyrics'].apply(lambda x: x.lower().split()).tolist()

# Train Word2Vec on our corpus (vector_size=128, window=5)
w2v_model = Word2Vec(sentences=tokenized, vector_size=128,
                     window=5, min_count=2, workers=4, epochs=10)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

# Build a song-level embedding = mean of its word vectors
def song_text_embedding(words, model):
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

text_embeddings = np.vstack(
    [song_text_embedding(tok, w2v_model) for tok in tokenized]
)
print(f"Text embeddings shape: {text_embeddings.shape}")  # (N, 128)

# =============================================
# 2b. AUDIO EMBEDDINGS  (PCA on MFCC stats)
# =============================================

# Extract only the numeric MFCC columns (skip id column)
mfcc_cols = [c for c in df.columns if c not in ['id', 'genre', 'lyrics']]
mfcc_matrix = df[mfcc_cols].values.astype(np.float32)

# Replace any NaN/Inf with 0 for safety
mfcc_matrix = np.nan_to_num(mfcc_matrix, nan=0.0, posinf=0.0, neginf=0.0)

# PCA: keep 95% of variance
pca = PCA(n_components=0.95, random_state=42)
audio_embeddings = pca.fit_transform(mfcc_matrix)
print(f"Audio PCA: {mfcc_matrix.shape[1]} dims → {audio_embeddings.shape[1]} dims "
      f"({pca.n_components_} components, 95% variance)")

# Store embeddings back in df for convenience
df['text_emb'] = list(text_embeddings)
df['audio_emb'] = list(audio_embeddings)
print("Embeddings stored in DataFrame ✓")


## 3. Οπτικοποίηση και Ανάλυση (Exploratory Data Analysis - EDA)
Ανάλυση των tags, word clouds, 2D projections (PCA/t-SNE) και Cosine Similarity.

## ΜΕΡΟΣ Β: Κατηγοριοποίηση & Clustering